# 3. Model Training

This notebook loads the preprocessed data and trains the model using the optimal hyperparameters identified in the previous step.

**Important Note:** Before running this notebook, you should update the `src/config.py` file with the best hyperparameters found by `02_hyperparameter_tuning.ipynb`. The training process below directly uses the values set in `config.py`.

**Key Steps:**
1.  **Load Processed Data:** Load the `.pt` file created by `01_data_preprocessing.ipynb`.
2.  **Initialize Model with Optimal Hyperparameters:** The model is initialized using parameters like `DROPOUT` and `WEIGHT_DECAY` from `src/config.py`.
3.  **Iterative Training:** Train the model for `NUM_RUNS` (e.g., 100) independent runs with different random seeds to ensure robustness.
4.  **Save Model Weights:** Save the state dictionary (`state_dict`) of each trained model to a separate `.pth` file for downstream analysis.

### 3.1. Import Libraries and Configuration


In [1]:
import sys
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
import pandas as pd
import os

# Add the project's 'src' directory to the Python path
sys.path.append('/data02/jaejoon/T2D_subtype_analysis/src')

# Import custom modules
import config
from models import MIC
from utils import set_seed, train_model

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### 3.2. Load Preprocessed Data


In [2]:
# Load the data saved from the previous notebook
data_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
processed_data = torch.load(data_path, weights_only=False)

input_genotype = processed_data['input_genotype']
input_proteome = processed_data['input_proteome']
input_metabolite = processed_data['input_metabolite']
output_clinical = processed_data['output_clinical']
clinical_df = processed_data['clinical_df']

# Create DataLoader
train_dataset = TensorDataset(input_genotype, input_proteome, input_metabolite, output_clinical)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE)

print("Data loaded successfully.")
print(f"Train dataset size: {len(train_dataset)}")


Data loaded successfully.
Train dataset size: 670


### 3.3. Train Models over Multiple Runs

To ensure the stability and robustness of our findings, we train the model 100 times with different random seeds. The weights of each model are saved for downstream analysis.


In [3]:
# Create directory to save models if it doesn't exist
config.MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for run in range(config.NUM_RUNS):
    # Set a different seed for each run for random initialization
    run_seed = 100+run
    set_seed(run_seed)
    
    print(f"--- Starting Run {run+1}/{config.NUM_RUNS} (Seed: {run_seed}) ---")

    # --- Dynamically define input dimensions from loaded data ---
    input_dims = {
        'genotype': input_genotype.shape[1],
        'proteome': input_proteome.shape[1],
        'metabolite': input_metabolite.shape[1]
    }

    # --- Initialize model ---
    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=config.DECODER_DIMS,
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=config.DROPOUT
    ).to(device)
    
    # Initialize optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=config.SCHEDULER_STEP_SIZE, gamma=config.SCHEDULER_GAMMA)
    
    # Train the model
    # The train_model function is imported from utils.py
    acc_list, loss_list, _ = train_model(
        model=model,
        clinical_df=clinical_df,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=config.EPOCHS
    )
    
    # Save the model's state dictionary
    model_save_path = config.MODEL_SAVE_DIR / f"mic_run_{run}.pth"
    torch.save(model.state_dict(), model_save_path)
    
    print(f"Run {run+1} complete. Final accuracy: {acc_list[-1]:.4f}")
    print(f"Model saved to: {model_save_path}\n")

print("All training runs completed.")

--- Starting Run 1/100 (Seed: 100) ---
Training started...


Epochs: 100%|██████████| 50/50 [00:06<00:00,  7.92it/s]


Epoch 050: | Loss: 0.0671 | Accuracy: 0.9000 | LR: 0.003487
Cluster distribution: Counter({np.int32(0): 286, np.int32(2): 166, np.int32(3): 152, np.int32(1): 66})
Training finished.
Run 1 complete. Final accuracy: 0.9000
Model saved to: /data02/jaejoon/T2D_subtype_analysis/outputs/models/mic_run_0.pth

--- Starting Run 2/100 (Seed: 101) ---
Training started...


Epochs: 100%|██████████| 50/50 [00:07<00:00,  6.94it/s]


Epoch 050: | Loss: 0.0658 | Accuracy: 0.8343 | LR: 0.003487
Cluster distribution: Counter({np.int32(1): 338, np.int32(0): 163, np.int32(2): 105, np.int32(3): 64})
Training finished.
Run 2 complete. Final accuracy: 0.8343
Model saved to: /data02/jaejoon/T2D_subtype_analysis/outputs/models/mic_run_1.pth

--- Starting Run 3/100 (Seed: 102) ---
Training started...


...

Epoch 050: | Loss: 0.0655 | Accuracy: 0.7612 | LR: 0.003487
Cluster distribution: Counter({np.int32(1): 298, np.int32(3): 204, np.int32(0): 109, np.int32(2): 59})
Training finished.
Run 98 complete. Final accuracy: 0.7612
Model saved to: /data02/jaejoon/T2D_subtype_analysis/outputs/models/mic_run_97.pth

--- Starting Run 99/100 (Seed: 198) ---
Training started...


Epochs: 100%|██████████| 50/50 [00:05<00:00,  9.23it/s]


Epoch 050: | Loss: 0.0583 | Accuracy: 0.8701 | LR: 0.003487
Cluster distribution: Counter({np.int32(0): 291, np.int32(3): 178, np.int32(1): 137, np.int32(2): 64})
Training finished.
Run 99 complete. Final accuracy: 0.8701
Model saved to: /data02/jaejoon/T2D_subtype_analysis/outputs/models/mic_run_98.pth

--- Starting Run 100/100 (Seed: 199) ---
Training started...


Epochs: 100%|██████████| 50/50 [00:05<00:00,  9.17it/s]


Epoch 050: | Loss: 0.0606 | Accuracy: 0.9015 | LR: 0.003487
Cluster distribution: Counter({np.int32(0): 271, np.int32(1): 197, np.int32(3): 171, np.int32(2): 31})
Training finished.
Run 100 complete. Final accuracy: 0.9015
Model saved to: /data02/jaejoon/T2D_subtype_analysis/outputs/models/mic_run_99.pth

All training runs completed.
